In [ ]:
def GetELEMENTS(PDB_protein, SDF_ligand, distance_cutoff=5.0):
# Function for the calculation of ELEMENTS
    Pairs = GetPLPairs(PDB_protein, SDF_ligand, distance_cutoff=distance_cutoff)
    ELEMENTS = [list(Pairs["ELEMENTS_PAIR"]).count(x) for x in PossibleELEMENTS]
    return ELEMENTS

In [35]:
import os
from tr_ec_pair import *
import glob
import pandas as pd

def make_elements_feature_from_dir(pdbid_dir, distance_cutoff):
    pdbid = os.path.basename(pdbid_dir)
    protein_pdb = os.path.join(pdbid_dir, f"{pdbid}_protein.pdb")
    ligand_sdf = os.path.join(pdbid_dir, f"{pdbid}_ligandMS.sdf")
    
    # Check if both required files exist before proceeding
    if not (os.path.exists(protein_pdb) and os.path.exists(ligand_sdf)):
        print(f"Warning: Missing files for {pdbid}, skipping...")
        return pdbid, None  # Return None if files are missing
    
    # Debugging: Check what GetELEMENTS returns
    result = GetELEMENTS(protein_pdb, ligand_sdf, distance_cutoff)
    print(f"Returned result for {pdbid}: {result}")

    # Adjust unpacking based on actual output
    if isinstance(result, tuple) and len(result) >= 1:
        elements_feature = result[0]  # Only take the first element
    else:
        elements_feature = result  # If result is not a tuple, use it as-is
    
    return pdbid, elements_feature

# Define parameters
distance_cutoff = 5.0
structure_dir = r"E:\NewProject\example_ref"

# Get list of PDB ID directories
pdbid_dirs = glob.glob(os.path.join(structure_dir, "*"))

# Process each directory
features = [make_elements_feature_from_dir(pdbid_dir, distance_cutoff) for pdbid_dir in pdbid_dirs]

# Remove None values (skipped entries due to missing files)
features = [f for f in features if f[1] is not None]

# Extract feature values and PDB IDs
feature_values = [f[1] for f in features]
pdbids = [f[0] for f in features]

# Define column names
columns = PossibleELEMENTS  # Ensure this variable is defined in your script

# Create and save DataFrame
elements_feature_df = pd.DataFrame(feature_values, index=pdbids, columns=columns)
elements_feature_df.index.name = 'PDB'

# Save the output
output_path = r'myex_elements_features.csv'
elements_feature_df.to_csv(output_path)

print(f"Feature extraction completed. Saved to {output_path}")


Returned result for 1a1e: [0, 97, 0, 0, 0, 3, 52, 11, 0, 0, 29, 0, 0, 0, 3, 24, 6, 0, 0, 25, 0, 0, 0, 2, 15, 3, 0, 0, 3, 0, 0, 0, 0, 2, 0, 0]
Returned result for 1b38: [0, 68, 0, 0, 0, 57, 82, 14, 0, 0, 7, 0, 0, 0, 6, 28, 7, 0, 0, 13, 0, 0, 0, 5, 30, 9, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Returned result for 1b55: [0, 25, 0, 0, 0, 0, 133, 33, 0, 0, 10, 0, 0, 0, 0, 51, 12, 0, 0, 13, 0, 0, 0, 0, 34, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Feature extraction completed. Saved to myex_elements_features.csv


In [37]:
from PDB_ECIF import generate_ecif_descriptors

data_path = "E:\\NewProject\\example_ref"
output_csv = "ECIF_pdb_descriptors.csv"

generate_ecif_descriptors(data_path, output_csv)


ECIF descriptors saved to ECIF_pdb_descriptors.csv


In [ ]:
import os
import pandas as pd
from rdkit import Chem
from rdkit.Chem import SDWriter
from SDF_ECIF import ECIFElements

def process_ligand_files(protein_pdb, ligand_sdf_folder, output_sdf_folder, distance_cutoff=6.0):
    if not os.path.exists(output_sdf_folder):
        os.makedirs(output_sdf_folder)
    
    ecif = ECIFElements()
    ligand_files = [f for f in os.listdir(ligand_sdf_folder) if f.endswith(".sdf")]
    
    for ligand_file in ligand_files:
        ligand_path = os.path.join(ligand_sdf_folder, ligand_file)
        mol = Chem.SDMolSupplier(ligand_path, removeHs=False)[0]
        
        if mol is None:
            print(f"Skipping {ligand_file}, could not read molecule.")
            continue
        
        ecif_descriptor = ecif.GetECIF(protein_pdb, ligand_path, distance_cutoff)
        elements_descriptor = ecif.GetELEMENTS(protein_pdb, ligand_path, distance_cutoff)
        
        descriptor_dict = {"ECIF_" + ecif.PossibleECIF[i]: ecif_descriptor[i] for i in range(len(ecif.PossibleECIF))}
        descriptor_dict.update({"ELEMENTS_" + ecif.PossibleELEMENTS[i]: elements_descriptor[i] for i in range(len(ecif.PossibleELEMENTS))})
        
        for key, value in descriptor_dict.items():
            mol.SetProp(key, str(value))
        
        output_path = os.path.join(output_sdf_folder, ligand_file)
        writer = SDWriter(output_path)
        writer.write(mol)
        writer.close()
        
        print(f"Processed {ligand_file}, saved descriptors to {output_path}")

if __name__ == "__main__":
    protein_pdb_file = "path/to/protein.pdb"  # Change to your protein PDB file path
    ligand_sdf_directory = "path/to/ligands/"  # Change to your ligand SDF directory
    output_sdf_directory = "path/to/output/"  # Change to your output directory
    process_ligand_files(protein_pdb_file, ligand_sdf_directory, output_sdf_directory)


# Get ECIF ELEMENTS

In [5]:
import os
import pandas as pd
from SDF_ECIF import ECIFElements

def process_elements_descriptors(base_path, output_csv="elements_descriptors.csv"):
    ecif = ECIFElements(atom_keys_file="E:/NewProject/data_ecif/PDB_Atom_Keys.csv")
    results = []
    
    # Iterate over all PDB folders
    for pdb_code in os.listdir(base_path):
        pdb_folder = os.path.join(base_path, pdb_code)
        if os.path.isdir(pdb_folder):
            protein_file = os.path.join(pdb_folder, f"{pdb_code}_protein.pdb")
            ligand_file = os.path.join(pdb_folder, f"{pdb_code}_ligandMS.sdf")
            
            if os.path.exists(protein_file) and os.path.exists(ligand_file):
                try:
                    elements_descriptor = ecif.GetELEMENTS(protein_file, ligand_file)
                    results.append([pdb_code] + elements_descriptor)
                    print(f"Processed: {pdb_code}")
                except Exception as e:
                    print(f"Error processing {pdb_code}: {e}")
            else:
                print(f"Missing files for {pdb_code}")
    
    # Save to CSV
    column_names = ["PDB_code"] + ecif.PossibleELEMENTS
    df = pd.DataFrame(results, columns=column_names)
    df.to_csv(output_csv, index=False)
    print(f"ELEMENTS descriptors saved to {output_csv}")

# Set your dataset path here
data_path = "E:/NewProject/example_ref"
process_elements_descriptors(data_path)


Processed: 1a1e
Processed: 1b38
Processed: 1b55
ELEMENTS descriptors saved to elements_descriptors.csv


# ECIF(Extended Connectivity Interacting Features)

In [3]:
import os
import pandas as pd
from SDF_ECIF import ECIFElements

def process_ecif_for_all(data_dir, atom_keys_file, output_csv):
    ecif = ECIFElements(atom_keys_file)
    all_data = []
    
    for pdb_code in os.listdir(data_dir):
        pdb_folder = os.path.join(data_dir, pdb_code)
        if not os.path.isdir(pdb_folder):
            continue  # Skip non-directory files
        
        protein_file = os.path.join(pdb_folder, f"{pdb_code}_protein.pdb")
        ligand_file = os.path.join(pdb_folder, f"{pdb_code}_ligandMS.sdf")
        
        if os.path.exists(protein_file) and os.path.exists(ligand_file):
            ecif_features = ecif.GetECIF(protein_file, ligand_file)
            all_data.append([pdb_code] + ecif_features)
        else:
            print(f"Missing files for {pdb_code}, skipping...")
    
    # Save to CSV
    column_names = ["PDB_CODE"] + ecif.PossibleECIF
    df = pd.DataFrame(all_data, columns=column_names)
    df.to_csv(output_csv, index=False)
    print(f"ECIF descriptors saved to {output_csv}")

# Define paths
data_directory = "E:/NewProject/example_ref"
atom_keys_path = "E:/NewProject/data_ecif/PDB_Atom_Keys.csv"
output_file = "E:/NewProject/data_ecif/ecif_descriptors.csv"


# Run ECIF extraction
process_ecif_for_all(data_directory, atom_keys_path, output_file)


ECIF descriptors saved to E:/NewProject/data_ecif/ecif_descriptors.csv
